# 02. DermAI Milestone 2: Model Training on Kaggle (GPU Acceleration)
### DermAI: Mobile Skin Lesion Screening & Decision Support System

This notebook runs deep learning training and evaluation for **Milestone 2** on Kaggle with a free GPU (NVIDIA T4 or P100).

---

### Workflow Overview:
1. **Verify GPU**: Check CUDA device availability and GPU memory (`nvidia-smi`).
2. **Install Libraries**: Install `timm`, `albumentations`, `segmentation-models-pytorch`, and `pyyaml`.
3. **Locate Data & Code**:
   - Images from Kaggle dataset: `skin-cancer-mnist-ham10000` (`HAM10000_images_part_1` & `HAM10000_images_part_2`).
   - Code & splits from attached dataset: `dermai-code` (mounted at `/kaggle/input/dermai-code/`).
4. **Fast Smoke Test**: Run 1 epoch with `--subset 50` on GPU to verify complete pipeline health in seconds.
5. **Full Model Training**:
   - Two-stage transfer learning (freeze head, then unfreeze all).
   - Automatic Mixed Precision (`torch.amp.autocast('cuda')`).
   - Epoch checkpointing to recover from timeouts with `--resume`.
6. **Validation Evaluation & Artifact Export**:
   - Generates validation confusion matrix and classification report.
   - Archives outputs (`best.pt`, `metrics.json`, plots) into a downloadable zip file.


## Step 1: Verify Hardware & GPU Acceleration
Ensure Kaggle accelerator is turned on: **Notebook options (right panel) -> Accelerator -> GPU T4 x2 or P100**.

In [ ]:
# Check GPU hardware and PyTorch CUDA support
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Device count: {torch.cuda.device_count()}")
else:
    print("WARNING: GPU not detected! Make sure you enabled Accelerator -> GPU in the right sidebar.")


## Step 2: Install Required Libraries

In [ ]:
!pip install --quiet timm>=0.9.16 albumentations>=1.4.0 segmentation-models-pytorch>=0.3.3 pyyaml>=6.0.1
print("Dependencies installed successfully.")


## Step 3: Setup Python Environment and Data Paths
We link the uploaded code dataset (`dermai-code`) to Python's module path (`sys.path` and `PYTHONPATH`) and locate the HAM10000 image folders.

In [ ]:
import os
import sys
import shutil
import glob
from pathlib import Path

# 1. Setup Python path for DermAI ml codebase
# When dermai-code is attached, it is mounted at /kaggle/input/dermai-code
dermai_code_roots = [
    "/kaggle/input/dermai-code",
    "/kaggle/input/dermai-code/dermai-code"
]

code_found = False
for cr in dermai_code_roots:
    if os.path.exists(cr) and os.path.exists(os.path.join(cr, "ml")):
        if cr not in sys.path:
            sys.path.insert(0, cr)
        os.environ["PYTHONPATH"] = f"{cr}:{os.environ.get('PYTHONPATH', '')}"
        print(f"Added DermAI code root to sys.path and PYTHONPATH: {cr}")
        code_found = True
        break

if not code_found:
    # Fallback search across all input folders
    for ml_dir in glob.glob("/kaggle/input/**/ml", recursive=True):
        parent_dir = str(Path(ml_dir).parent)
        if parent_dir not in sys.path:
            sys.path.insert(0, parent_dir)
        os.environ["PYTHONPATH"] = f"{parent_dir}:{os.environ.get('PYTHONPATH', '')}"
        print(f"Discovered DermAI codebase at: {parent_dir}")
        code_found = True
        break

if not code_found:
    print("WARNING: 'dermai-code' dataset not found in /kaggle/input/! Please attach your uploaded code dataset.")

# 2. Locate HAM10000 image folders
ham_root_candidates = [
    "/kaggle/input/skin-cancer-mnist-ham10000",
    "/kaggle/input/skin-cancer-mnist-ham10000/skin-cancer-mnist-ham10000"
]

ham_root = None
for hr in ham_root_candidates:
    if os.path.exists(hr):
        ham_root = hr
        break

if ham_root:
    img_part1 = os.path.join(ham_root, "HAM10000_images_part_1")
    img_part2 = os.path.join(ham_root, "HAM10000_images_part_2")
    print(f"HAM10000 image folders:")
    print(f"  Part 1 ({os.path.exists(img_part1)}): {img_part1}")
    print(f"  Part 2 ({os.path.exists(img_part2)}): {img_part2}")
else:
    print("WARNING: 'skin-cancer-mnist-ham10000' dataset not found! Please attach it from Kaggle Datasets.")

# 3. Locate splits.csv and copy to /kaggle/working/data/
# splits.csv lives inside the ml/ subfolder in the uploaded dataset
splits_candidates = [
    "/kaggle/input/dermai-code/ml/splits.csv",
    "/kaggle/input/dermai-code/dermai-code/ml/splits.csv",
    "/kaggle/input/dermai-code/splits.csv",          # fallback (old layout)
    "/kaggle/input/dermai-code/dermai-code/splits.csv"
] + glob.glob("/kaggle/input/**/splits.csv", recursive=True)

splits_csv_path = None
for sc in splits_candidates:
    if os.path.isfile(sc):
        splits_csv_path = sc
        break

os.makedirs("/kaggle/working/data", exist_ok=True)
working_splits_csv = "/kaggle/working/data/splits.csv"

if splits_csv_path:
    shutil.copy2(splits_csv_path, working_splits_csv)
    print(f"Copied splits.csv to: {working_splits_csv}")
else:
    print("WARNING: splits.csv not found in attached inputs!")


## Step 4: Fast GPU Smoke Test (--subset 50)
Runs 1 epoch with 50 stratified samples to verify GPU execution, AMP, and metrics in seconds before starting full training.

In [ ]:
# Smoke Test Parameters
MODEL_NAME = "resnet50"  # Choices: resnet50, efficientnet_b0, densenet121, mobilenetv3_large_100
IMG_PART1 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
IMG_PART2 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2"
SPLITS_CSV = "/kaggle/working/data/splits.csv"

print(">>> Starting 50-sample GPU Smoke Test <<<")
!python -m ml.training.train \
    --model {MODEL_NAME} \
    --images_root {IMG_PART1} {IMG_PART2} \
    --splits_csv {SPLITS_CSV} \
    --output_dir /kaggle/working/experiments/{MODEL_NAME}_smoke \
    --epochs 1 \
    --freeze_epochs 1 \
    --subset 50 \
    --batch_size 16 \
    --device auto \
    --num_workers 2

print("\n[OK] Smoke test completed successfully!")


## Step 5: Full Model Training (25 Epochs)
Runs full training with two-stage transfer learning, AdamW, cosine annealing, and class-weighted cross-entropy.
Saves checkpoints after every epoch to recover from timeouts with `--resume`.

In [ ]:
# Full Training Parameters
EPOCHS = 25
FREEZE_EPOCHS = 3
BATCH_SIZE = 32
LR = 3e-4
LR_FINE_TUNE = 3e-5
LOSS = "ce_weighted"  # Options: ce_weighted, focal
OUTPUT_DIR = f"/kaggle/working/experiments/{MODEL_NAME}"

print(f">>> Running Full Training for {MODEL_NAME} ({EPOCHS} Epochs) <<<")
!python -m ml.training.train \
    --model {MODEL_NAME} \
    --images_root {IMG_PART1} {IMG_PART2} \
    --splits_csv {SPLITS_CSV} \
    --output_dir {OUTPUT_DIR} \
    --epochs {EPOCHS} \
    --freeze_epochs {FREEZE_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --lr_fine_tune {LR_FINE_TUNE} \
    --loss {LOSS} \
    --scheduler cosine \
    --patience 7 \
    --device auto \
    --num_workers 4


## Step 6: Review Validation Metrics & Confusion Matrix
Displays the validation classification metrics and confusion matrix generated by the training run.

In [ ]:
from IPython.display import Image, display
import json

metrics_file = os.path.join(OUTPUT_DIR, "metrics.json")
if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        metrics = json.load(f)
    print("=== Validation Metrics ===")
    val_data = metrics.get("val", {})
    for k, v in val_data.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        elif isinstance(v, dict):
            pass
        else:
            print(f"  {k}: {v}")

# Display plots
for plot_name in ["val_confusion_matrix.png", "training_curves.png"]:
    plot_path = os.path.join(OUTPUT_DIR, plot_name)
    if os.path.exists(plot_path):
        print(f"\nDisplaying: {plot_name}")
        display(Image(filename=plot_path))


## Step 7: Package Artifacts for Direct Download
Zips the model experiment folder so you can download `best.pt`, `metrics.json`, and plots with one click from Kaggle's **Output** panel (right sidebar).

In [ ]:
import shutil

zip_filename = f"/kaggle/working/{MODEL_NAME}_results"
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)
print(f"Created archive: {zip_filename}.zip")
print(f"Archive size: {os.path.getsize(zip_filename + '.zip') / (1024 * 1024):.2f} MB")
print("You can download this zip file directly from Kaggle's Output panel on the right sidebar.")
